# 딥러닝의 개념과 작동 원리 2

## 학습방법

지도학습, 비지도학습, 자기지도학습, 강화학습

자기지도학습: 대규모 레이블 없는 데이터에 활용하는 학습 방법. BERT와 GPT가 있다.
> GPT를 지도학습으로 학습 시키는 것은 불가능하다. 학습하는 양이 수조개인데, 그것 하나하나 레이블(정답)을 줄 수 없다.  
> 그래서 GPT는 next word prediction으로, BERT는 mask를 채우는 방식으로 학습한다.
>
> 이 두 방식 모두 정답이 필요 없지만, 데이터 자체에서 지도 신호를 생성하기 때문에 마치 지도학습과 같은 학습이 이뤄지는 것이다


강화학습: 챗 지피티도 강화학습 한다! -> 환경과 상호작용하면서 보상을 최대화하는 학습법. 시행착오를 통해 최적의 행동 전략을 학습한다. 

지도학습을 생각해 보면,

익히 알다싶이 test, valid, train으로 데이터를 나눈다. 


이때 train 데이터 안에서 일어나는 일을 살펴보자

## 학습(training) 관련 용어

배치크기: 한 번에 모델에 넣는 즉, 한번에 학습하는 데이터 수(이때 나눈 조각조각을 미니배치라고 한다)  

이터레이션: 1 에폭을 끝내기 위한 배치 처리 횟수로, 매 이터레이션 마다 가중치(W)가 업데이트 된다(즉, 이터레이션은 하나의 배치를 사용하여 순전파 -> 손실 계산 -> 역전파 -> 가중치 업데이트를 마치는 한 번의 과정이다. 결국 이터레이션 크기 만큼 가중치를 업데이트 한다. )

에포크: '전체' 데이퍼 1회 순전파 & 역전파 완료.
> 1에폭이 끝나면 검증용 데이터로 검증을 한다. 안좋네? 그럼 다시 2에폭 시작. 정해진 에폭이 모두 끝나거나 허용치에 도달하면 test data로 최종 모델의 성능을 평가한다. 




좀 더 생각해 보면, 순전파와 역전파의 한 사이클이 1 이터레이션에서 일어난다. 
> iteration (반복): 하나의 배치를 사용하여 순전파 -> 손실 계산 -> 역전파 -> 가중치 업데이트를 마치는 한 번의 과정
>
> 

## 순전파 

입력층→ 은닉층→ 출력층 방향으로 한방향 연산
각층에서"선형변환(Wx+b) → 활성화 함수" 반복->  
최종 출력층에서 예측값(y-hat) 생성→ 이 예측값과
실제값의 차이를 손실함수로 계산.  
학습 초기에는 가중치가 무작위이므로 예측이 엉터리  
→ 역전파로 가중치를 조정해야함

$$Y = XW + B$$

이게 가장 간결한 공식이지만, 이거 하나하나 처리하지 않지!


$$\left[\begin{array}{cc} y_{11} & y_{12} \\ y_{21} & y_{22} \\ y_{31} & y_{32} \\ y_{41} & y_{42} \end{array}\right] = \left[\begin{array}{ccc} x_{11} & x_{12} & x_{13} \\ x_{21} & x_{22} & x_{23} \\ x_{31} & x_{32} & x_{33} \\ x_{41} & x_{42} & x_{43} \end{array}\right] \times \left[\begin{array}{cc} w_1 & w_4 \\ w_2 & w_5 \\ w_3 & w_6 \end{array}\right] + \left[\begin{array}{cc} b_1 & b_2 \end{array}\right]$$




X: 입력행렬(4,3) 크기 즉, 4개의 샘플과 3개의 특성   
W: 가중치 행렬 (3,2) 크기 즉, 2개의 출력으로 연결  
B: 편향 벡터(각 출력 노드에 더해지는 값)  
Y: 출력 행렬 (4,2) 크기 즉, 4개 샘플에 대한 2개씩의 결과값(예측 또는 분류..)

*이때 B는 행렬의 크기가 다르지만, 컴퓨터가 알아서 모든 샘플에 행을 element wies 하게 더해준다. 즉, 브로드캐스팅!!

컴퓨터는 메모리를 아끼기 위해 실제로 $B$를 4번 복사해서 저장하지는 않지만, 논리적으로는 다음과 같이 행을 늘려서 계산한다. 


$$B = \left[\begin{array}{cc} b_1 & b_2 \end{array}\right] \quad \xrightarrow{\text{Broadcasting}} \quad \left[\begin{array}{cc} b_1 & b_2 \\ b_1 & b_2 \\ b_1 & b_2 \\ b_1 & b_2 \end{array}\right]$$



행렬 곱셈의 원리에 따라, 첫 번째 샘플의 첫 번째 출력($y_{11}$)이 나오는 과정은 다음과 같다.

$$y_{11} = (x_{11} \cdot w_1 + x_{12} \cdot w_2 + x_{13} \cdot w_3) + b_1$$



*순전파(Forward Propagation)와 FFNN의 차이점*

FFNN은 구조를 지칭하는 용어이다(정보가 앞에서 뒤로만 흐르도록 설계된 신경망의 구조)

순전파는 연산 과정을 지칭하는 용어이다. 입력층에 데이터가 들어와서 가중치와 곱해지고 (선형변환) 그리고 활성화 함수를 거쳐 (비선형 변환) 최종 출력층 까지 전달되는 일련의 계산 과정을 의미하는 것이다. 

RNN은 FFNN 구조는 아니지만, 입력을 받아 결과를 내는 단계에서는 똑같이 순전파(forward propagation)을 수행한다. 

## 역전파 (가중치 업데이트) 


*역전파에서 중요한 것은 w를 고친다는 것이다. input x, 각 레이어의 output z, 활성화 함수 h 모두 아니라 w가 대상이다!*

dL/dW = dL/dy * dy/dz * dz/dW  
(연쇄법칙으로 각 가중치에 대한 기울기 계산)  
→ 이 기울기 방향으로 가중치를 조정

nabla(나블라)는 $\nabla$
-> 미분하라! 라는 연산자임. 근데, 벡터 대상으로 내리는 명령임!
> 나블라는 그 자체로 어떤 숫자를 의미하는 것이 아니라, "각 성분별로 편미분을 해서 벡터로 묶어라"라는 동작을 압축한 기호

 $\nabla L$이라는 표현은 **'모든 가중치에 대한 편미분 모음'**을 의미힌다. 즉, 존재하는 가중치가 w1 w2 w3라면. 


$$\nabla L = \left( \frac{\partial L}{\partial w_1}, \frac{\partial L}{\partial w_2}, \frac{\partial L}{\partial w_3} \right)$$


우리는 오차를 줄여야 하므로, 나블라가 가리키는 방향의 **반대($-\nabla L$)**로 가중치를 이동시키는 것이죠.

1. 개별 가중치에 집중할 때 (Scalar 표기)

 **"특정 가중치 하나(w_i)"**가 오차에 미치는 영향력을 나타낼 때 쓴다.
 
 $$w_{new} = w_{old} - \eta \cdot \frac{\partial Loss}{\partial w}$$

장점: "이 가중치 하나가 범인이다!"라고 지목해서 설명하기 좋습니다.의미: $w$라는 변수 하나에 대한 미분(편미분) 값임을 명시합니다.


2. 모든 가중치를 한 번에 다룰 때 (Vector/Gradient 표기)

나블라 기호를 쓰는 식은 "가중치 덩어리(벡터/행렬)" 전체를 한꺼번에 업데이트할 때 쓴다.

$$W_{new} = W_{old} - \eta \cdot \nabla_W Loss$$

장점: 수식이 매우 간결해진다. 수만 개의 가중치를 일일이 나열할 필요가 없다. 
의미: $\nabla_W Loss$
>  모든 가중치들에 대한 편미분 값들을 한데 모아놓은 **기울기 벡터(Gradient Vector)**를 뜻합니다.

## 손실함수

MSE -> 회귀문제  
BCE -> 이진분류 문제(출력층에서 시그모이드를 사용하겠지)  
CCE -> 다중분류 문제(츨력층에서 소프트맥스를 사용하겠지)



## 글 정리 및 최종 결론

이 글에서는 딥러닝의 여러 학습 방법을 구분하고, 훈련 데이터 안에서 하나의 미니배치가 순전파·손실 계산·역전파·가중치 업데이트를 거치는 과정을 살펴보았다. 또한 배치 크기, 이터레이션, 에포크의 관계와 행렬 연산, 브로드캐스팅, 기울기 표기, 문제 유형별 손실함수를 연결하여 신경망이 실제로 학습되는 한 사이클을 정리했다.

### 학습 방법의 구분

지도학습은 입력과 정답 레이블의 관계를 학습하고, 비지도학습은 명시적인 정답 없이 데이터의 구조와 패턴을 찾는다. 자기지도학습은 레이블이 없는 대규모 데이터에서 데이터 자체로 학습 신호를 만든다. GPT의 다음 단어 예측과 BERT의 마스크 복원은 원자료로부터 예측 대상을 구성한다는 점에서 자기지도학습의 대표적인 방식이다. 강화학습은 환경과 상호작용하면서 얻는 보상을 최대화하는 행동 전략을 시행착오를 통해 학습한다.

### 훈련 데이터에서 반복되는 학습 단위

전체 데이터는 일반적으로 훈련·검증·테스트 데이터로 나눈다. 모델은 훈련 데이터로 가중치를 학습하고, 검증 데이터로 학습 과정과 모델 선택을 점검하며, 마지막에 테스트 데이터로 최종 일반화 성능을 평가한다.

| 용어 | 의미 | 가중치 업데이트와의 관계 |
| --- | --- | --- |
| 배치 크기 | 한 번에 모델에 입력하는 데이터 수다. | 하나의 미니배치를 구성한다. |
| 이터레이션 | 미니배치 하나로 순전파부터 가중치 업데이트까지 수행하는 한 번의 과정이다. | 이터레이션마다 한 번 업데이트한다. |
| 에포크 | 전체 훈련 데이터를 한 차례 모두 사용한 상태다. | 한 에포크에는 여러 이터레이션이 포함된다. |

따라서 배치 크기가 B이고 훈련 사례 수가 N이라면 한 에포크에는 대략 N/B개의 이터레이션이 필요하다. 각 이터레이션에서 현재 미니배치로 모델의 예측과 오차를 계산하고 가중치를 수정한다. 이 과정을 여러 에포크에 걸쳐 반복하면서 손실을 줄인다.

### 순전파와 행렬 연산

순전파는 입력층에서 은닉층을 거쳐 출력층으로 계산이 진행되는 연산 과정이다. 각 층은 기본적으로 선형 변환 `Y = XW + B`와 활성화 함수 적용을 반복한다. 입력행렬 X가 `(4, 3)`, 가중치행렬 W가 `(3, 2)`라면 행렬 곱의 결과 Y는 `(4, 2)`가 되어 네 개의 사례마다 두 개의 출력값을 만든다.

편향 B가 `(1, 2)` 형태여도 각 사례의 출력에 같은 편향이 더해진다. 컴퓨터는 B를 실제로 여러 번 복사하지 않고 브로드캐스팅을 이용하여 행마다 같은 값을 적용한다. 이처럼 행렬 연산과 브로드캐스팅을 사용하면 사례와 가중치를 하나씩 계산하지 않고 하나의 배치를 동시에 처리할 수 있다.

FFNN은 정보가 입력에서 출력 방향으로 흐르도록 설계된 신경망의 구조를 뜻하고, 순전파는 그 구조에서 예측값을 계산하는 과정을 뜻한다. 두 용어는 관련되어 있지만 구조와 연산 과정이라는 서로 다른 층위의 개념이다. RNN도 FFNN 구조는 아니지만 주어진 입력으로 출력을 계산하는 단계에서는 순전파를 수행한다.

### 역전파와 가중치 업데이트

순전파로 만든 예측값과 실제값의 차이를 손실함수로 계산하면, 역전파는 연쇄법칙을 이용하여 각 가중치가 손실에 얼마나 영향을 주었는지 구한다. 역전파의 직접적인 수정 대상은 입력 X나 중간 출력 자체가 아니라 학습 가능한 가중치 W다.

특정 가중치 하나의 갱신은 `w_new = w_old − η × ∂Loss/∂w`로 표현할 수 있다. 모든 가중치를 한꺼번에 다루면 `W_new = W_old − η × ∇_W Loss`로 표현한다. ∇는 각 가중치에 대한 편미분을 계산하여 하나의 기울기 벡터나 행렬로 묶으라는 연산자다. 두 식은 서로 다른 알고리즘이 아니라 하나의 가중치를 보느냐 전체 가중치를 보느냐에 따른 표기 차이다.

### 문제 유형과 손실함수

| 문제 유형 | 대표 손실함수 | 출력층과의 연결 |
| --- | --- | --- |
| 회귀 | MSE | 연속적인 예측값과 실제값의 차이를 측정한다. |
| 이진분류 | BCE | 시그모이드 출력과 정답 사이의 차이를 측정한다. |
| 다중분류 | CCE | 소프트맥스가 만든 클래스별 확률과 정답 사이의 차이를 측정한다. |

손실함수는 단순한 평가 지표가 아니라 역전파가 미분할 학습 목표다. 문제에 맞지 않는 손실함수를 선택하면 모델이 줄여야 할 오차의 기준도 잘못 설정된다.

### 최종 결론

딥러닝의 훈련은 하나의 거대한 계산이 아니라 작은 학습 사이클의 반복이다. 모델은 미니배치를 입력받아 순전파로 예측값을 만들고, 문제에 맞는 손실함수로 오차를 계산하며, 역전파로 모든 가중치의 기울기를 구한 뒤 가중치를 갱신한다. 이 한 사이클이 한 이터레이션이며, 전체 훈련 데이터를 모두 사용할 때 한 에포크가 완성된다.

결국 신경망 학습을 이해하는 핵심은 데이터의 반복 단위와 계산의 방향을 함께 보는 데 있다. 배치·이터레이션·에포크는 학습이 얼마나 반복되는지를 설명하고, 순전파는 예측이 만들어지는 방향을, 손실함수는 줄여야 할 목표를, 역전파는 각 가중치를 어떻게 수정할지 설명한다. 이 요소들이 하나의 연속된 과정으로 작동할 때 모델은 무작위로 초기화된 가중치를 데이터에 맞는 값으로 점차 변화시킨다.